# Đo một epoch và tính ngân sách GPU
Bật GPU và Internet trên Colab/Kaggle. Upload repo hiện tại có `tools/gpu_budget.py`.
Tải dữ liệu và kiểm tra MD5 bằng các ô có sẵn trong `starter/lab_day2.ipynb`.
Notebook này chỉ đo thời gian train/val; không chạy test và không thay thế pipeline bài lab.

In [ ]:
import sys, subprocess
from pathlib import Path
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'timm', 'pandas', 'pillow'], check=True)
import torch
assert torch.cuda.is_available(), 'Bật GPU trong cài đặt phiên chạy.'
print('GPU:', torch.cuda.get_device_name(0), '| torch:', torch.__version__)

In [ ]:
# Sửa các đường dẫn theo nơi upload repo/dataset.
REPO_DIR = Path('/content/K4-Track4-Day2-Deeplearning-Advance')
IMAGES_DIR = Path('/content/data/images')
LABELS_DIR = Path('/content/data/labels')
OUTPUT_DIR = Path('/content/gpu_budget') if Path('/content').is_dir() else Path('/kaggle/working/gpu_budget')
GPU_HOURS = None  # Nhập hạn mức tổng thật; None = kế hoạch chưa chốt.
BATCH_SIZE = 32
EPOCHS = 10
SELECTED = ['resnet50']  # Giả định để tính giờ; có thể nhập 2 backbone đã đo.
FINAL_BACKBONE = SELECTED[0]
assert (REPO_DIR / 'tools/gpu_budget.py').is_file(), 'Đặt đúng REPO_DIR và upload file mới.'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
# Khoảng 5 epoch thực: một epoch mỗi backbone. Không có số giả định.
subprocess.run([sys.executable, str(REPO_DIR / 'tools/gpu_budget.py'), 'measure',
    '--images-dir', str(IMAGES_DIR), '--labels-dir', str(LABELS_DIR),
    '--batch-size', str(BATCH_SIZE), '--epochs', str(EPOCHS),
    '--out', str(OUTPUT_DIR / 'profile.json')], check=True)

In [ ]:
# Có thể chạy lại ô này để so kế hoạch 1 và 2 backbone, không cần đo lại.
command = [sys.executable, str(REPO_DIR / 'tools/gpu_budget.py'), 'plan',
    '--profile', str(OUTPUT_DIR / 'profile.json'), '--selected', *SELECTED,
    '--final-backbone', FINAL_BACKBONE, '--epochs', str(EPOCHS),
    '--reserve', '0.25', '--extra-hours', '1', '--out', str(OUTPUT_DIR / 'plan.md')]
if GPU_HOURS is not None:
    command += ['--gpu-hours', str(GPU_HOURS)]
subprocess.run(command, check=True)
print('Giữ lại profile.json và plan.md trong:', OUTPUT_DIR)